<div style='background:#8B0000; padding:20px; border-radius:8px; margin-bottom:10px'>
<h1 style='color:white; text-align:center; margin:0'>Reconocimiento de Patrones</h1>
<h2 style='color:#ffcccc; text-align:center; margin:5px 0 0 0'>Challenge 1 — Procesamiento y Visualización de Datos Biomédicos</h2>
<p style='color:#ffaaaa; text-align:center; margin:5px 0 0 0'>Ingeniería Biomédica · UPCH · 2026-1</p>
</div>

## Contexto clínico

Se te proporciona el dataset **Heart Failure Prediction** (Kaggle, fedesoriano, 2021), que contiene registros clínicos de **918 pacientes** con variables demográficas, de laboratorio y de electrocardiograma. La variable objetivo es `HeartDisease` (0 = sin enfermedad, 1 = con enfermedad).

Tu misión como analista: **explorar, limpiar, preprocesar y construir la matriz X e y** lista para ingresar a un modelo de ML clásico, aplicando todos los conceptos vistos en Clase 2.

---
> **Dataset:** `heart.csv`  
> **Fuente:** fedesoriano. (2021). *Heart Failure Prediction Dataset*. Kaggle. https://www.kaggle.com/fedesoriano/heart-failure-prediction

<div style='background:#e8f5e9; padding:8px 14px; border-left:4px solid #2e7d32; border-radius:4px; margin:8px 0'>
⏱️ <b>EN CLASE</b> — completa esto durante la sesión
</div>

---
## Ejercicio 1 — Diseño del pipeline (conceptual)

> **Antes de escribir una línea de código**, debes entender qué vas a hacer y por qué.

### 1.1 — Diagrama de bloques del pipeline

Completa el siguiente diagrama describiendo qué ocurre en cada etapa:

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│                 │     │                 │     │                 │
│   heart.csv     │────▶│   [???]         │────▶│   [???]         │
│   (raw data)    │     │                 │     │                 │
│                 │     │  ¿Qué haces?:   │     │  ¿Qué haces?:   │
│  N=918, d=11    │     │  __________     │     │  __________     │
└─────────────────┘     └─────────────────┘     └────────┬────────┘
                                                          │
                                                          ▼
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│                 │     │                 │     │                 │
│  X ∈ ℝ^(N×d)    │◀────│  [???]          │◀────│     [???]       │
│  y ∈ {0,1}^N    │     │                 │     │                 │
│                 │     │  ¿Qué haces?:   │     │  ¿Qué haces?:   │
│  listo p/ ML    │     │  __________     │     │  __________     │
└─────────────────┘     └─────────────────┘     └─────────────────┘
```

**📝 Tu respuesta (edita esta celda):**
- **???:** ___
- **???:** ___
- **???** ___
- **???:** ___

### 1.2 — Representación matemática del pipeline

Completa las dimensiones y operaciones matemáticas para este dataset:

**Punto de partida — el dataset crudo:**

$$\text{heart.csv} \rightarrow \text{DataFrame} \in \mathbb{R}^{N \times (d+1)}$$

donde $N = $ ___ pacientes y $d+1 = $ ___ columnas (features + target).

---

**Paso 1 — Separar features y target:**

$$\text{DataFrame} \xrightarrow{\text{split}} X_{\text{raw}} \in \mathbb{R}^{N \times \, \underline{\hspace{1cm}}}, \quad y \in \{0,1\}^{\underline{\hspace{1cm}}}$$

---

**Paso 2 — Codificación de variables categóricas (One-Hot):**

El dataset tiene columnas categóricas: `Sex`, `ChestPainType`, `RestingECG`, `ExerciseAngina`, `ST_Slope`.

¿Cuántas columnas nuevas generan? Completa:
(hint: drop_first)
| Variable | Categorías | Columnas One-Hot |
|---|---|---|
| Sex | M, F | ___ |
| ChestPainType | ATA, NAP, ASY, TA | ___ |
| RestingECG | Normal, ST, LVH | ___ |
| ExerciseAngina | Y, N | ___ |
| ST_Slope | Up, Flat, Down | ___ |
| **Total nuevas columnas** | | **___** |

$$X_{\text{raw}} \in \mathbb{R}^{N \times d_{\text{raw}}} \xrightarrow{\text{One-Hot}} X_{\text{enc}} \in \mathbb{R}^{N \times \underline{\hspace{1cm}}}$$

---

**Paso 3 — Escalado Z-score:**

$$x'_j = \frac{x_j - \mu_j}{\sigma_j} \quad \Rightarrow \quad X_{\text{enc}} \xrightarrow{\text{Z-score}} X \in \mathbb{R}^{N \times \underline{\hspace{1cm}}}$$

¿Sobre qué conjunto calculas $\mu_j$ y $\sigma_j$? ___. ¿Por qué? ___.

---

**📝 Tu respuesta (edita esta celda):** completa los espacio en blanco de arriba.

### 1.3 — Predicción sobre el desbalance

Antes de cargar los datos, responde:

1. ¿Esperarías que este dataset esté desbalanceado? ¿Por qué clínicamente?
2. Si el 60% de los pacientes tiene `HeartDisease=1`, ¿qué accuracy obtendría un clasificador que **siempre** predice 1?
3. ¿Sería útil ese clasificador? ¿Por qué?

**📝 Tu respuesta (edita esta celda):**
1. ___
2. ___
3. ___

In [ ]:
# ── Conexión con Google Drive ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')


---
## Ejercicio 2 — Carga y EDA

### Paso 2.0 — Setup

In [ ]:
# Colores institucionales UPCH
UPCH_RED   = '#8B0000'
UPCH_BLUE  = '#1565C0'
UPCH_GRAY  = '#4A4A4A'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split

# Ruta del dataset — ajusta según tu carpeta en Drive
DATA_PATH = Path('COLOCA LA RUTA')

print('Librerías cargadas correctamente ✓')

### Paso 2.1 — Carga y primera inspección

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f'Shape del dataset: {df.shape}')
print(f'\nPrimeras filas:')
df.head()

In [ ]:
# Tipos de datos y valores nulos
print('Tipos de datos:')
print(df.dtypes)
print(f'\nValores nulos por columna:')
print(df.isnull().sum())

In [ ]:
# Estadísticas descriptivas
df.describe()

### Paso 2.2 — Distribución de clases (desbalance)

> Recuerda lo que predijiste en Ejercicio 1.3. ¿Acertaste?

In [ ]:
# --- COMPLETA EL CÓDIGO ---
# Visualiza la distribución de la variable objetivo HeartDisease
# Incluye: conteo por clase, porcentaje, y gráfico de barras con colores UPCH

conteo = df['HeartDisease'].value_counts()

# TODO 1: calcula el porcentaje de cada clase (usa conteo y len(df))
pct = ___

print('Distribución de clases:')
print(f'  Sin enfermedad (0): {conteo[0]} pacientes ({pct[0]:.1f}%)')
print(f'  Con enfermedad (1): {conteo[1]} pacientes ({pct[1]:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 4))

# TODO 2: grafica las barras — usa ax.bar(), colores [UPCH_BLUE, UPCH_RED]
___

ax.set_title('Distribución de clases — Heart Disease', fontweight='bold', color=UPCH_RED)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Sin enfermedad (0)', 'Con enfermedad (1)'])
ax.set_ylabel('Número de pacientes')

# TODO 3: agrega el valor numérico encima de cada barra con ax.text()
___

plt.tight_layout()
plt.savefig('distribucion_clases.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ ¿Hay desbalance? ¿Qué implicaciones clínicas tiene?')


### Paso 2.3 — Visualización exploratoria de features numéricas

In [ ]:
# Features numéricas del dataset
features_num = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for i, feat in enumerate(features_num):
    # Fila 0: ya está resuelta — histograma por clase
    for clase, color, label in [(0, UPCH_BLUE, 'Sin enf.'), (1, UPCH_RED, 'Con enf.')]:
        axes[0, i].hist(
            df[df['HeartDisease'] == clase][feat],
            bins=30, alpha=0.6, color=color, label=label, density=True
        )
    axes[0, i].set_title(feat, fontweight='bold')
    axes[0, i].legend(fontsize=7)
    axes[0, i].set_xlabel('Valor')

    # --- COMPLETA EL CÓDIGO ---
    # TODO: Fila 1 — boxplot separado por clase en axes[1, i]
    # Pasos:
    #   1. Crea una lista con los valores de cada clase: [valores_clase0, valores_clase1]
    #   2. Llama a axes[1, i].boxplot() con patch_artist=True y labels=['Sin enf.', 'Con enf.']
    #   3. Colorea bp['boxes'][0] con UPCH_BLUE+'88' y bp['boxes'][1] con UPCH_RED+'88'
    #   4. Pon título: f'{feat} — boxplot'
    ___

fig.suptitle('EDA — Features numéricas por clase (Heart Disease)',
             fontsize=13, fontweight='bold', color=UPCH_RED)
plt.tight_layout()
plt.savefig('eda_numericas.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ ¿Qué features parecen tener mayor poder discriminativo entre clases?')


### Paso 2.4 — Visualización de features categóricas

In [ ]:
features_cat = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for i, feat in enumerate(features_cat):
    # --- COMPLETA EL CÓDIGO ---
    # TODO 1: calcula la tasa de enfermedad por categoría
    #         (% de pacientes con HeartDisease=1 para cada valor de feat)
    #         pista: df.groupby(feat)
    tasa = ___

    # TODO 2: asigna color UPCH_RED si tasa > 50%, UPCH_BLUE si no
    colores = ___

    # TODO 3: grafica las barras y la línea de referencia al 50%
    ___

    axes[i].set_title(feat, fontweight='bold')
    axes[i].set_ylabel('% con HeartDisease' if i == 0 else '')
    axes[i].set_ylim(0, 100)

    # TODO 4: agrega el valor (ej: "72%") encima de cada barra con axes[i].text()
    ___

fig.suptitle('EDA — Tasa de enfermedad cardíaca por variable categórica',
             fontsize=12, fontweight='bold', color=UPCH_RED)
plt.tight_layout()
plt.savefig('eda_categoricas.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ ¿Qué categorías tienen mayor riesgo de enfermedad cardíaca?')


---
## Ejercicio 3 — Limpieza de datos

### Paso 3.1 — Detección y manejo de outliers

> **⚠️ Recuerda:** en biomédica, un outlier puede ser un caso clínico real. Analiza antes de eliminar.

In [ ]:
from scipy import stats

# --- COMPLETA EL CÓDIGO ---
# TODO: para cada feature numérica, calcula el z-score y cuenta cuántos
#       valores tienen |z| > 3. Imprime el resultado por feature.
# Pista: stats.zscore(), np.abs(), suma de booleanos

print('Detección de outliers (|z-score| > 3):')
print('-' * 45)
for feat in features_num:
    ___


In [ ]:
# El dataset tiene Cholesterol = 0 en varios pacientes (imposible fisiológicamente)
# Esto es un error de adquisición, no un caso clínico real

print(f'Pacientes con Cholesterol = 0: {(df["Cholesterol"] == 0).sum()}')
print(f'Pacientes con RestingBP  = 0: {(df["RestingBP"]  == 0).sum()}')

df_clean = df.copy()

# --- COMPLETA EL CÓDIGO ---
# TODO: reemplaza los valores Cholesterol == 0 por NaN
# Pista: usa df_clean.loc[]
___

print(f'\nValores NaN en Cholesterol tras limpieza: {df_clean["Cholesterol"].isnull().sum()}')
print('→ La imputación se realizará en el Paso 4 (después del split, sobre train)')


---
## Ejercicio 4 — Preprocesamiento

### Paso 4.1 — Train/test split

> **Regla fundamental:** separar primero, preprocesar después. Nunca al revés — eso es **data leakage**.

In [ ]:
TARGET = 'HeartDisease'
FEATURES_NUM = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak', 'FastingBS']
FEATURES_CAT = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

X_raw = df_clean[FEATURES_NUM + FEATURES_CAT].copy()
y     = df_clean[TARGET].values

# --- COMPLETA EL CÓDIGO ---
# TODO: split 80/20, estratificado por y, random_state=42
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    ___
)

print(f'Train: {X_train_raw.shape[0]} muestras')
print(f'Test:  {X_test_raw.shape[0]} muestras')
print(f'\nProporción HeartDisease=1 en train: {y_train.mean():.3f}')
print(f'Proporción HeartDisease=1 en test:  {y_test.mean():.3f}')
print('→ Con stratify, ambas proporciones deben ser similares ✓')


### Paso 4.2 — Imputación de valores faltantes (sobre train)

In [ ]:
# --- COMPLETA EL CÓDIGO ---
# TODO 1: calcula la mediana de Cholesterol SOLO sobre el train set
mediana_colesterol_train = ___

X_train_imp = X_train_raw.copy()
X_test_imp  = X_test_raw.copy()

# TODO 2: imputa NaN en train y test usando la mediana del train
X_train_imp['Cholesterol'] = ___
X_test_imp['Cholesterol']  = ___

print(f'Mediana Cholesterol (train): {mediana_colesterol_train:.1f} mg/dL')
print(f'NaN restantes en train: {X_train_imp.isnull().sum().sum()}')
print(f'NaN restantes en test:  {X_test_imp.isnull().sum().sum()}')


### Paso 4.3 — Codificación de variables categóricas (One-Hot)

> Recuerda tu tabla del Ejercicio 1.2 — ¿coincide el número de columnas?

In [ ]:
# One-Hot Encoding de variables categóricas
# drop_first=True para evitar multicolinealidad (dummy variable trap)

# --- COMPLETA EL CÓDIGO ---
# TODO: aplica pd.get_dummies a train y test usando FEATURES_CAT
X_train_enc = ___
X_test_enc  = ___

# Alinear columnas (por si alguna categoría no aparece en test)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

print(f'Shape antes de One-Hot: {X_train_raw.shape}')
print(f'Shape después de One-Hot: {X_train_enc.shape}')
print(f'\nColumnas generadas:')
print(list(X_train_enc.columns))


### Paso 4.4 — Escalado Z-score

$$x'_j = \frac{x_j - \mu_j}{\sigma_j}$$

> **Anti-leakage:** `fit` solo sobre train, `transform` sobre train y test.

In [ ]:
scaler = StandardScaler()

# --- COMPLETA EL CÓDIGO ---
# TODO: fit SOLO sobre train, transform sobre train y test
X_train = ___
X_test  = ___

print(f'X_train: {X_train.shape}  dtype: {X_train.dtype}')
print(f'X_test:  {X_test.shape}')
print(f'\nMedia de X_train (debe ser ≈ 0): {X_train.mean():.4f}')
print(f'Std  de X_train (debe ser ≈ 1): {X_train.std():.4f}')
print(f'\nMedia de X_test (no será exactamente 0): {X_test.mean():.4f}')


---
## Ejercicio 5 — Verificación final y resumen del pipeline

### Paso 5.1 — El dataset final en formato ML

In [ ]:
# --- COMPLETA EL CÓDIGO ---
# TODO: imprime un resumen del dataset final mostrando:
#   - shape de X_train, X_test
#   - shape de y_train, y_test
#   - número y % de positivos en train y test
#   - el pipeline aplicado en una línea

print('=' * 50)
print('  RESUMEN — Dataset listo para ML clásico')
print('=' * 50)
___


### Paso 5.2 — Visualización Before vs After del preprocesamiento

In [ ]:
features_plot = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
idx_features  = [list(X_train_enc.columns).index(f) for f in features_plot]

fig, axes = plt.subplots(2, 5, figsize=(18, 6))

for i, (feat, idx) in enumerate(zip(features_plot, idx_features)):
    # Fila 0: antes del escalado — ya resuelta
    axes[0, i].hist(X_train_enc[feat].values, bins=30, color=UPCH_BLUE, alpha=0.7)
    axes[0, i].set_title(f'{feat}\nAntes', fontsize=9, fontweight='bold')
    axes[0, i].set_xlabel('Valor original')

    # --- COMPLETA EL CÓDIGO ---
    # TODO: Fila 1 — histograma de X_train[:, idx] después del escalado
    ___

fig.suptitle('Preprocesamiento — Antes vs. Después del escalado Z-score',
             fontsize=12, fontweight='bold', color=UPCH_RED)
plt.tight_layout()
plt.savefig('before_after_scaling.png', dpi=150, bbox_inches='tight')
plt.show()


### Paso 5.3 — Mapa de correlaciones

In [ ]:
cols_corr = FEATURES_NUM + [TARGET]
corr_matrix = df_clean[cols_corr].corr()

fig, ax = plt.subplots(figsize=(8, 6))

# --- COMPLETA EL CÓDIGO ---
# TODO: genera el heatmap con sns.heatmap
# Parámetros sugeridos: cmap='RdBu_r', annot=True, fmt='.2f', center=0, square=True
___

ax.set_title('Mapa de correlación — Features numéricas + HeartDisease',
             fontweight='bold', color=UPCH_RED)
plt.tight_layout()
plt.savefig('correlacion.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ ¿Qué feature tiene mayor correlación (positiva o negativa) con HeartDisease?')


---
<div style='background:#f5f5f5; padding:15px; border-left:5px solid #8B0000; border-radius:4px'>
<b>Entrega:</b> Sube tu notebook ejecutado (.ipynb con outputs) a tu carpeta de GitHub del curso.<br>
<b>Nombre del archivo:</b> <code>Challenge1_ApellidoNombreDeAmbos.ipynb</code><br>
<b>Fecha límite:</b> antes de Clase 3
</div>